# Evaluation on DeepSeek-LLM-7B using CounterBench for reasoning tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 13.7 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import classification_report

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
model_name = "deepseek-ai/deepseek-llm-7b-chat"

In [5]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model= AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [7]:
def get_prompt(instruction,questions,contexts):
  prompts=[]
  for i in range(len(questions)):
    prompt = f"{instruction}\nContext:{contexts[i]}\nQuestion:{questions[i]}\nAnswer:"
    prompts.append(prompt)
  return prompts

In [8]:
def training(prompts, num_tokens=1):
    predictions = []
    for i, prompt in enumerate(prompts):
        chat = [
            {"role": "user", "content": prompt}
        ]


        formatted_prompt = tokenizer.apply_chat_template(
            chat,
            tokenize=False,
            add_generation_prompt=True
        )


        inputs = tokenizer(
            formatted_prompt,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to("cuda")

        with torch.no_grad():
            output_tokens = model.generate(
                **inputs,
                max_new_tokens=num_tokens,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.1,
                num_beams=1,
            )

        input_length = inputs["input_ids"].shape[1]
        new_tokens = output_tokens[0][input_length:]

        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        print(f"Iteration {i}")
        print(f"Model Output: {response}\n")

        predictions.append(response)

    return predictions

In [9]:
ds = load_dataset(
    "json",
    data_files=[
        "hf://datasets/CounterBench/CounterBench/data_balanced_alpha_V1.json",
        "hf://datasets/CounterBench/CounterBench/data_balanced_backdoor_V2.json"
    ]
)

print(ds)
print(ds["train"].features)

data_balanced_alpha_V1.json: 0.00B [00:00, ?B/s]

data_balanced_backdoor_V2.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'given_info', 'answer', 'type', 'question_id', 'meta'],
        num_rows: 1200
    })
})
{'question': Value('string'), 'given_info': Value('string'), 'answer': Value('string'), 'type': Value('string'), 'question_id': Value('string'), 'meta': {'graph_id': Value('string'), 'model_id': Value('int64'), 'query_type': Value('string'), 'rung': Value('int64'), 'story_id': Value('string')}}


In [10]:
dataset = ds["train"].to_pandas()

In [11]:
dataset

,question,given_info,answer,type,question_id,meta
0,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf, Blaf causes Tr...",no,basic,0,"{'graph_id': 'graph5', 'model_id': 0, 'query_t..."
1,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz, Razz causes Pe...",no,basic,200,"{'graph_id': 'graph5', 'model_id': 1, 'query_t..."
2,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank, Vank causes Scu...",no,basic,400,"{'graph_id': 'graph5', 'model_id': 2, 'query_t..."
3,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn, Blorn causes P...",no,basic,600,"{'graph_id': 'graph5', 'model_id': 3, 'query_t..."
4,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes Splee, Splee causes Bl...",no,basic,800,"{'graph_id': 'graph5', 'model_id': 4, 'query_t..."
...,...,...,...,...,...,...
1195,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Blaf causes Ziklo, Blaf causes Tr...",no,backdoor,39,"{'graph_id': 'graph9', 'model_id': 195, 'query..."
1196,Would Zlim occur if not Glent instead of Glent?,"We know that Razz causes Glent, Razz causes Pe...",no,backdoor,239,"{'graph_id': 'graph9', 'model_id': 196, 'query..."
1197,Would Klep occur if not Praf instead of Praf?,"We know that Vank causes Praf, Vank causes Scu...",no,backdoor,439,"{'graph_id': 'graph9', 'model_id': 197, 'query..."
1198,Would Jext occur if not Fizo instead of Fizo?,"We know that Blorn causes Fizo, Blorn causes P...",no,backdoor,639,"{'graph_id': 'graph9', 'model_id': 198, 'query..."


In [12]:
dataset['type'].value_counts()

,count
type,
basic,250
conditional,250
joint,250
nested,250
backdoor,200


In [13]:
instruction = "Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else."

## Basic Questions

In [14]:
dataset_basic = dataset[dataset['type']=="basic"]

In [15]:
dataset_basic

,question,given_info,answer,type,question_id,meta
0,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf, Blaf causes Tr...",no,basic,0,"{'graph_id': 'graph5', 'model_id': 0, 'query_t..."
1,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz, Razz causes Pe...",no,basic,200,"{'graph_id': 'graph5', 'model_id': 1, 'query_t..."
2,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank, Vank causes Scu...",no,basic,400,"{'graph_id': 'graph5', 'model_id': 2, 'query_t..."
3,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn, Blorn causes P...",no,basic,600,"{'graph_id': 'graph5', 'model_id': 3, 'query_t..."
4,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes Splee, Splee causes Bl...",no,basic,800,"{'graph_id': 'graph5', 'model_id': 4, 'query_t..."
...,...,...,...,...,...,...
495,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf and Trune, Trun...",no,basic,99,"{'graph_id': 'graph9', 'model_id': 495, 'query..."
496,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz and Pex, Pex ca...",no,basic,299,"{'graph_id': 'graph9', 'model_id': 496, 'query..."
497,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank and Scud, Scud c...",no,basic,499,"{'graph_id': 'graph9', 'model_id': 497, 'query..."
498,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn and Plim, Plim ...",no,basic,699,"{'graph_id': 'graph9', 'model_id': 498, 'query..."


In [16]:
questions = dataset_basic["question"].values.tolist()
contexts = dataset_basic["given_info"].values.tolist()
true_answers = dataset_basic["answer"].values.tolist()

In [17]:
prompts = get_prompt(instruction,questions,contexts)

In [18]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.\nQuestion:Would Lumbo occur if not Ziklo instead of Ziklo?\nAnswer:'

In [19]:
predictions = training(prompts)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Iteration 0
Model Output: No

Iteration 1
Model Output: No

Iteration 2
Model Output: No

Iteration 3
Model Output: No

Iteration 4
Model Output: No

Iteration 5
Model Output: No

Iteration 6
Model Output: Yes

Iteration 7
Model Output: No

Iteration 8
Model Output: No

Iteration 9
Model Output: No

Iteration 10
Model Output: Yes

Iteration 11
Model Output: No

Iteration 12
Model Output: No

Iteration 13
Model Output: No

Iteration 14
Model Output: No

Iteration 15
Model Output: Yes

Iteration 16
Model Output: No

Iteration 17
Model Output: No

Iteration 18
Model Output: No

Iteration 19
Model Output: No

Iteration 20
Model Output: Yes

Iteration 21
Model Output: No

Iteration 22
Model Output: Yes

Iteration 23
Model Output: No

Iteration 24
Model Output: No

Iteration 25
Model Output: Yes

Iteration 26
Model Output: No

Iteration 27
Model Output: Yes

Iteration 28
Model Output: No

Iteration 29
Model Output: Yes

Iteration 30
Model Output: Yes

Iteration 31
Model Output: No

Iteration

In [20]:
predictions_new = [elem.lower() for elem in predictions]

In [21]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.56      0.80      0.66       125
         yes       0.65      0.37      0.47       125

    accuracy                           0.58       250
   macro avg       0.60      0.58      0.56       250
weighted avg       0.60      0.58      0.56       250



## Conditional Questions

In [22]:
dataset_conditional = dataset[dataset['type']=="conditional"]

In [23]:
dataset_conditional

,question,given_info,answer,type,question_id,meta
25,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes not Blaf, Blaf and T...",yes,conditional,5,"{'graph_id': 'graph5', 'model_id': 25, 'query_..."
26,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes not Razz, Razz and P...",yes,conditional,205,"{'graph_id': 'graph5', 'model_id': 26, 'query_..."
27,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes not Vank, Vank and Sc...",yes,conditional,405,"{'graph_id': 'graph5', 'model_id': 27, 'query_..."
28,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes not Blorn, Blorn and ...",yes,conditional,605,"{'graph_id': 'graph5', 'model_id': 28, 'query_..."
29,Would Wrox occur if not Nuv instead of Nuv?,"We know that Nuv causes not Splee, Splee and B...",yes,conditional,805,"{'graph_id': 'graph5', 'model_id': 29, 'query_..."
...,...,...,...,...,...,...
435,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Ziklo causes Blaf and Trune, Blaf...",no,conditional,87,"{'graph_id': 'graph7', 'model_id': 435, 'query..."
436,Would Zlim occur if not Glent instead of Glent?,"We know that Glent causes Razz and Pex, Razz a...",no,conditional,287,"{'graph_id': 'graph7', 'model_id': 436, 'query..."
437,Would Klep occur if not Praf instead of Praf?,"We know that Praf causes Vank and Scud, Vank a...",no,conditional,487,"{'graph_id': 'graph7', 'model_id': 437, 'query..."
438,Would Jext occur if not Fizo instead of Fizo?,"We know that Fizo causes Blorn and Plim, Blorn...",no,conditional,687,"{'graph_id': 'graph7', 'model_id': 438, 'query..."


In [24]:
questions = dataset_conditional["question"].values.tolist()
contexts = dataset_conditional["given_info"].values.tolist()
true_answers = dataset_conditional["answer"].values.tolist()

In [25]:
prompts = get_prompt(instruction,questions,contexts)

In [26]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune\nQuestion:Would Lumbo occur if not Ziklo instead of Ziklo?\nAnswer:'

In [27]:
predictions = training(prompts)

Iteration 0
Model Output: No

Iteration 1
Model Output: No

Iteration 2
Model Output: No

Iteration 3
Model Output: Yes

Iteration 4
Model Output: No

Iteration 5
Model Output: Yes

Iteration 6
Model Output: No

Iteration 7
Model Output: Yes

Iteration 8
Model Output: Yes

Iteration 9
Model Output: Yes

Iteration 10
Model Output: No

Iteration 11
Model Output: No

Iteration 12
Model Output: No

Iteration 13
Model Output: No

Iteration 14
Model Output: No

Iteration 15
Model Output: Yes

Iteration 16
Model Output: No

Iteration 17
Model Output: Yes

Iteration 18
Model Output: Yes

Iteration 19
Model Output: Yes

Iteration 20
Model Output: Yes

Iteration 21
Model Output: No

Iteration 22
Model Output: Yes

Iteration 23
Model Output: Yes

Iteration 24
Model Output: No

Iteration 25
Model Output: Yes

Iteration 26
Model Output: No

Iteration 27
Model Output: Yes

Iteration 28
Model Output: No

Iteration 29
Model Output: No

Iteration 30
Model Output: Yes

Iteration 31
Model Output: No

Ite

In [28]:
predictions_new = [elem.lower() for elem in predictions]

In [29]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.62      0.68      0.65       125
         yes       0.64      0.58      0.61       125

    accuracy                           0.63       250
   macro avg       0.63      0.63      0.63       250
weighted avg       0.63      0.63      0.63       250



## Joint Questions

In [30]:
dataset_joint = dataset[dataset['type']=="joint"]

In [31]:
dataset_joint

,question,given_info,answer,type,question_id,meta
500,Would Lumbo occur if not Ziklo and not Blaf?,"We know that Ziklo causes Blaf and Vork, Blaf ...",yes,joint,100,"{'graph_id': 'graph5', 'model_id': 500, 'query..."
501,Would Zlim occur if not Glent and not Razz?,"We know that Glent causes Razz and Zurn, Razz ...",yes,joint,300,"{'graph_id': 'graph5', 'model_id': 501, 'query..."
502,Would Klep occur if not Praf and not Vank?,"We know that Praf causes Vank and Wrenk, Vank ...",yes,joint,500,"{'graph_id': 'graph5', 'model_id': 502, 'query..."
503,Would Jext occur if not Fizo and not Blorn?,"We know that Fizo causes Blorn and Quaz, Blorn...",yes,joint,700,"{'graph_id': 'graph5', 'model_id': 503, 'query..."
504,Would Wrox occur if not Nuv and not Splee?,"We know that Nuv causes Splee and Druk, Splee ...",yes,joint,900,"{'graph_id': 'graph5', 'model_id': 504, 'query..."
...,...,...,...,...,...,...
745,Would Lumbo occur if not Ziklo and not Sline?,"We know that Ziklo causes Blaf, Blaf causes Tr...",yes,joint,149,"{'graph_id': 'graph7', 'model_id': 745, 'query..."
746,Would Zlim occur if not Glent and not Melf?,"We know that Glent causes Razz, Razz causes Pe...",yes,joint,349,"{'graph_id': 'graph7', 'model_id': 746, 'query..."
747,Would Klep occur if not Praf and not Yobb?,"We know that Praf causes Vank, Vank causes Scu...",yes,joint,549,"{'graph_id': 'graph7', 'model_id': 747, 'query..."
748,Would Jext occur if not Fizo and not Skul?,"We know that Fizo causes Blorn, Blorn causes P...",yes,joint,749,"{'graph_id': 'graph7', 'model_id': 748, 'query..."


In [32]:
questions = dataset_joint["question"].values.tolist()
contexts = dataset_joint["given_info"].values.tolist()
true_answers = dataset_joint["answer"].values.tolist()

In [33]:
prompts = get_prompt(instruction,questions,contexts)

In [34]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes Blaf and Vork, Blaf causes not Trune, Trune or Vork causes Lumbo.\nQuestion:Would Lumbo occur if not Ziklo and not Blaf?\nAnswer:'

In [35]:
predictions = training(prompts)

Iteration 0
Model Output: Yes

Iteration 1
Model Output: Yes

Iteration 2
Model Output: Yes

Iteration 3
Model Output: Yes

Iteration 4
Model Output: Yes

Iteration 5
Model Output: Yes

Iteration 6
Model Output: Yes

Iteration 7
Model Output: Yes

Iteration 8
Model Output: Yes

Iteration 9
Model Output: No

Iteration 10
Model Output: No

Iteration 11
Model Output: No

Iteration 12
Model Output: No

Iteration 13
Model Output: No

Iteration 14
Model Output: No

Iteration 15
Model Output: Yes

Iteration 16
Model Output: Yes

Iteration 17
Model Output: Yes

Iteration 18
Model Output: Yes

Iteration 19
Model Output: Yes

Iteration 20
Model Output: Yes

Iteration 21
Model Output: Yes

Iteration 22
Model Output: Yes

Iteration 23
Model Output: Yes

Iteration 24
Model Output: No

Iteration 25
Model Output: Yes

Iteration 26
Model Output: Yes

Iteration 27
Model Output: Yes

Iteration 28
Model Output: Yes

Iteration 29
Model Output: Yes

Iteration 30
Model Output: Yes

Iteration 31
Model Output

In [36]:
predictions_new = [elem.lower() for elem in predictions]

In [37]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.82      0.49      0.61       125
         yes       0.64      0.90      0.74       125

    accuracy                           0.69       250
   macro avg       0.73      0.69      0.68       250
weighted avg       0.73      0.69      0.68       250



## Nested Questions

In [38]:
dataset_nested = dataset[dataset['type']=="nested"]

In [39]:
dataset_nested

,question,given_info,answer,type,question_id,meta
750,"Assume not Ziklo, and based on this assumption...","We know that Ziklo causes Blaf, Trune and Vork...",yes,nested,150,"{'graph_id': 'graph9', 'model_id': 750, 'query..."
751,"Assume not Glent, and based on this assumption...","We know that Glent causes Razz, Pex and Zurn, ...",yes,nested,350,"{'graph_id': 'graph9', 'model_id': 751, 'query..."
752,"Assume not Praf, and based on this assumption,...","We know that Praf causes Vank, Scud and Wrenk,...",yes,nested,550,"{'graph_id': 'graph9', 'model_id': 752, 'query..."
753,"Assume not Fizo, and based on this assumption,...","We know that Fizo causes Blorn, Plim and Quaz,...",yes,nested,750,"{'graph_id': 'graph9', 'model_id': 753, 'query..."
754,"Assume not Nuv, and based on this assumption, ...","We know that Nuv causes Splee, Blen and Druk, ...",yes,nested,950,"{'graph_id': 'graph9', 'model_id': 754, 'query..."
...,...,...,...,...,...,...
995,"Assume not Ziklo, and based on this assumption...","We know that Ziklo causes Blaf and Trune, Trun...",yes,nested,199,"{'graph_id': 'graph8', 'model_id': 995, 'query..."
996,"Assume not Glent, and based on this assumption...","We know that Glent causes Razz and Pex, Pex ca...",yes,nested,399,"{'graph_id': 'graph8', 'model_id': 996, 'query..."
997,"Assume not Praf, and based on this assumption,...","We know that Praf causes Vank and Scud, Scud c...",yes,nested,599,"{'graph_id': 'graph8', 'model_id': 997, 'query..."
998,"Assume not Fizo, and based on this assumption,...","We know that Fizo causes Blorn and Plim, Plim ...",yes,nested,799,"{'graph_id': 'graph8', 'model_id': 998, 'query..."


In [40]:
questions = dataset_nested["question"].values.tolist()
contexts = dataset_nested["given_info"].values.tolist()
true_answers = dataset_nested["answer"].values.tolist()

In [41]:
prompts = get_prompt(instruction,questions,contexts)

In [42]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Ziklo causes Blaf, Trune and Vork, not Vork causes Sline, Sline causes Frim, Frim or Vork causes Qado, Qado causes Jurf, and Jurf causes Lumbo.\nQuestion:Assume not Ziklo, and based on this assumption, further suppose not Sline. Would Lumbo occur?\nAnswer:'

In [43]:
predictions = training(prompts)

Iteration 0
Model Output: No

Iteration 1
Model Output: No

Iteration 2
Model Output: No

Iteration 3
Model Output: No

Iteration 4
Model Output: Yes

Iteration 5
Model Output: Yes

Iteration 6
Model Output: Yes

Iteration 7
Model Output: No

Iteration 8
Model Output: No

Iteration 9
Model Output: No

Iteration 10
Model Output: No

Iteration 11
Model Output: No

Iteration 12
Model Output: No

Iteration 13
Model Output: No

Iteration 14
Model Output: No

Iteration 15
Model Output: No

Iteration 16
Model Output: No

Iteration 17
Model Output: No

Iteration 18
Model Output: No

Iteration 19
Model Output: No

Iteration 20
Model Output: No

Iteration 21
Model Output: No

Iteration 22
Model Output: No

Iteration 23
Model Output: No

Iteration 24
Model Output: No

Iteration 25
Model Output: No

Iteration 26
Model Output: No

Iteration 27
Model Output: No

Iteration 28
Model Output: No

Iteration 29
Model Output: No

Iteration 30
Model Output: No

Iteration 31
Model Output: No

Iteration 32
Mo

In [44]:
predictions_new = [elem.lower() for elem in predictions]

In [45]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.53      0.93      0.68       125
         yes       0.72      0.18      0.29       125

    accuracy                           0.56       250
   macro avg       0.63      0.56      0.48       250
weighted avg       0.63      0.56      0.48       250



## Backdoor Questions

In [46]:
dataset_backdoor = dataset[dataset['type']=="backdoor"]

In [47]:
dataset_backdoor

,question,given_info,answer,type,question_id,meta
1000,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Blaf causes Ziklo, Ziklo or Blaf ...",yes,backdoor,0,"{'graph_id': 'graph5', 'model_id': 0, 'query_t..."
1001,Would Zlim occur if not Glent instead of Glent?,"We know that Razz causes Glent, Glent or Razz ...",yes,backdoor,200,"{'graph_id': 'graph5', 'model_id': 1, 'query_t..."
1002,Would Klep occur if not Praf instead of Praf?,"We know that Vank causes Praf, Praf or Vank ca...",yes,backdoor,400,"{'graph_id': 'graph5', 'model_id': 2, 'query_t..."
1003,Would Jext occur if not Fizo instead of Fizo?,"We know that Blorn causes Fizo, Fizo or Blorn ...",yes,backdoor,600,"{'graph_id': 'graph5', 'model_id': 3, 'query_t..."
1004,Would Wrox occur if not Nuv instead of Nuv?,"We know that Splee causes Nuv, Nuv or Splee ca...",yes,backdoor,800,"{'graph_id': 'graph5', 'model_id': 4, 'query_t..."
...,...,...,...,...,...,...
1195,Would Lumbo occur if not Ziklo instead of Ziklo?,"We know that Blaf causes Ziklo, Blaf causes Tr...",no,backdoor,39,"{'graph_id': 'graph9', 'model_id': 195, 'query..."
1196,Would Zlim occur if not Glent instead of Glent?,"We know that Razz causes Glent, Razz causes Pe...",no,backdoor,239,"{'graph_id': 'graph9', 'model_id': 196, 'query..."
1197,Would Klep occur if not Praf instead of Praf?,"We know that Vank causes Praf, Vank causes Scu...",no,backdoor,439,"{'graph_id': 'graph9', 'model_id': 197, 'query..."
1198,Would Jext occur if not Fizo instead of Fizo?,"We know that Blorn causes Fizo, Blorn causes P...",no,backdoor,639,"{'graph_id': 'graph9', 'model_id': 198, 'query..."


In [48]:
questions = dataset_backdoor["question"].values.tolist()
contexts = dataset_backdoor["given_info"].values.tolist()
true_answers = dataset_backdoor["answer"].values.tolist()

In [49]:
prompts = get_prompt(instruction,questions,contexts)

In [50]:
prompts[0]

'Instruction: Answer the question strictly using the causal facts provided. Do not introduce any new information. For counterfactuals, simulate the scenario along the causal chain step by step. Only output the final answer as Yes or No. Do not write anything else.\nContext:We know that Blaf causes Ziklo, Ziklo or Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. Blaf~ Bern(0.2). We observed Trune\nQuestion:Would Lumbo occur if not Ziklo instead of Ziklo?\nAnswer:'

In [51]:
predictions = training(prompts)

Iteration 0
Model Output: Yes

Iteration 1
Model Output: No

Iteration 2
Model Output: No

Iteration 3
Model Output: Yes

Iteration 4
Model Output: Yes

Iteration 5
Model Output: Yes

Iteration 6
Model Output: No

Iteration 7
Model Output: No

Iteration 8
Model Output: No

Iteration 9
Model Output: Yes

Iteration 10
Model Output: Yes

Iteration 11
Model Output: No

Iteration 12
Model Output: No

Iteration 13
Model Output: No

Iteration 14
Model Output: Yes

Iteration 15
Model Output: Yes

Iteration 16
Model Output: Yes

Iteration 17
Model Output: Yes

Iteration 18
Model Output: Yes

Iteration 19
Model Output: Yes

Iteration 20
Model Output: Yes

Iteration 21
Model Output: Yes

Iteration 22
Model Output: Yes

Iteration 23
Model Output: Yes

Iteration 24
Model Output: Yes

Iteration 25
Model Output: Yes

Iteration 26
Model Output: Yes

Iteration 27
Model Output: Yes

Iteration 28
Model Output: Yes

Iteration 29
Model Output: Yes

Iteration 30
Model Output: Yes

Iteration 31
Model Output:

In [52]:
predictions_new = [elem.lower() for elem in predictions]

In [53]:
print(classification_report(true_answers,predictions_new,zero_division=0))

              precision    recall  f1-score   support

          no       0.67      0.11      0.20       105
         yes       0.49      0.94      0.64        95

    accuracy                           0.51       200
   macro avg       0.58      0.53      0.42       200
weighted avg       0.58      0.51      0.41       200

